In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

# Encontrar la carpeta raíz del proyecto
def find_project_root(start: Path) -> Path:
    """Busca hacia arriba la carpeta que contiene `src` y `settings.gradle.kts`."""
    for p in [start, *start.parents]:
        if (p / "src").is_dir() and (p / "settings.gradle.kts").exists():
            return p
    raise FileNotFoundError("No se encontró la raíz del proyecto")

project_root = find_project_root(Path.cwd()).resolve()
modelos_dir = project_root / "modelos"
modelos_dir.mkdir(parents=True, exist_ok=True)

class AudioPreprocessONNX(nn.Module):
    def __init__(self):
        super().__init__()
        # PARÁMETROS SINCRONIZADOS CON proy1.ipynb (src/preprocess/config.py)
        self.sample_rate = 16000
        
        # STFT Parameters (DEBEN coincidir con AudioPreprocessConfig)
        self.n_fft = 512          # ✓ Sincronizado con proy1
        self.win_length = 400     # ✓ Sincronizado con proy1 (25ms @ 16kHz)
        self.hop_length = 160     # ✓ Sincronizado con proy1 (10ms @ 16kHz)
        
        # Mel Parameters
        self.n_mels = 64          # ✓ Sincronizado con proy1
        self.f_min = 20.0         # ✓ Sincronizado con proy1
        self.f_max = 8000.0       # ✓ Sincronizado con proy1 (sample_rate / 2)
        self.eps = 1e-6           # ✓ Sincronizado con proy1
        
        # Output image size (como en proy1.ipynb IMG_SIZE = 128)
        self.output_size = 128
        
        # Precompute Hann window (win_length = 400)
        self.register_buffer("window", torch.hann_window(self.win_length, dtype=torch.float32))
        
        # Precompute mel filterbank (64 filtros)
        from torchaudio.functional import melscale_fbanks
        fb = melscale_fbanks(
            n_freqs=(self.n_fft // 2) + 1,      # 257 para n_fft=512
            f_min=self.f_min,
            f_max=self.f_max,
            n_mels=self.n_mels,                 # 64, no 128
            sample_rate=self.sample_rate,
            norm='slaney',
            mel_scale='htk'
        ).transpose(0, 1)  # -> (n_mels=64, freq_bins=257)
        
        self.register_buffer("fb", fb.float())
        
    def forward(self, waveform):
        # waveform: (batch, samples)
        
        # PASO 1: STFT con Hann Window (ventana de 400 muestras)
        stft = torch.stft(
            waveform,
            n_fft=self.n_fft,           # 512
            hop_length=self.hop_length,  # 160
            win_length=self.win_length,  # 400
            window=self.window,          # Hann window
            center=True,
            pad_mode="reflect",
            return_complex=True,
        )
        # stft shape: (batch, freq_bins=257, frames)
        
        # PASO 2: Power Spectrogram (magnitud al cuadrado)
        power = (stft.real**2 + stft.imag**2).clamp_min(0.0)
        # power shape: (batch, 257, frames)
        
        # PASO 3: Aplicar Mel Filterbank
        # fb shape: (64, 257), power shape: (batch, 257, frames)
        mel = torch.matmul(self.fb, power)  # (batch, 64, frames)
        
        # PASO 4: Log (natural log, como en proy1)
        mel = torch.log(mel + self.eps)
        
        # PASO 5: Normalización per-sample Z-score (estandarización)
        # Normalizar: (x - mean) / std (SINCRONIZADO con proy1)
        batch_size = mel.shape[0]
        for b in range(batch_size):
            mean = mel[b].mean()
            std = mel[b].std(unbiased=False).clamp_min(self.eps)
            mel[b] = (mel[b] - mean) / std
        
        # mel shape: (batch, 64, variable_frames)
        
        # PASO 6: Agregar dimensión de canal (para que sea (batch, 1, 64, frames))
        mel = mel.unsqueeze(1)  # (batch, 1, 64, frames)
        
        # PASO 7: Redimensionar a 128x128 (como en proy1 IMG_SIZE=128)
        # Interpolate de (batch, 1, 64, frames) a (batch, 1, 128, 128)
        mel = F.interpolate(
            mel, 
            size=(self.output_size, self.output_size),
            mode='bilinear',
            align_corners=False
        )  # (batch, 1, 128, 128)
        
        # PASO 8: Normalización final (mean=0.5, std=0.5 como en proy1)
        # raw_transform = transforms.Normalize(mean=[0.5], std=[0.5])
        # Esta normalización convierte [0, 1] → [-1, 1]
        mel = (mel - 0.5) / 0.5
        
        # Output shape: (batch, 1, 128, 128) — exactamente lo que espera el modelo
        return mel

model = AudioPreprocessONNX()
model.eval()

# Test run y ONNX export
dummy_input = torch.randn(1, 16000, dtype=torch.float32)
onnx_path = modelos_dir / 'audio_preprocessor.onnx'

print(f"Exportando a: {onnx_path}")
print(f"\n✓ PIPELINE COMPLETO SINCRONIZADO CON proy1.ipynb:")
print(f"  1. STFT: n_fft={model.n_fft}, win_length={model.win_length}, hop_length={model.hop_length}")
print(f"  2. Mel: {model.n_mels} bins (20-8000 Hz)")
print(f"  3. Log: log(mel + {model.eps})")
print(f"  4. Z-score normalization per-sample")
print(f"  5. Resize to {model.output_size}x{model.output_size}")
print(f"  6. Normalize (mean=0.5, std=0.5)")
print(f"\n  Output shape: (batch, 1, 128, 128) ✓ Compatible con Modelo B")

torch.onnx.export(
    model, 
    dummy_input, 
    str(onnx_path),
    export_params=True,
    opset_version=17,          
    do_constant_folding=True, 
    input_names=['audio'], 
    output_names=['mel_spectrogram'],
    dynamic_axes={
        'audio': {0: 'batch_size', 1: 'num_samples'}, 
        'mel_spectrogram': {0: 'batch_size'}
    }
)

print(f"\n Pre-processor exportado exitosamente a {onnx_path}")
print(f"   Input: (batch_size, 16000)")
print(f"   Output: (batch_size, 1, 128, 128)")